In [2]:
import Pkg
Pkg.add("JuMP")
Pkg.add("HiGHS")


    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
Precompiling packages...
              ✗ CPLEX
  0 dependencies successfully precompiled in 3 seconds. 59 already precompiled.

The following 1 direct dependency failed to precompile:

CPLEX 

Failed to precompile CPLEX [a076750e-1247-5638-91d2-ce28b192dca0] to "/home/husted42/.julia/compiled/v1.12/CPLEX/jl_oKZPiw".
ERROR: LoadError: CPLEX not properly installed. Please run Pkg.build("CPLEX")
Stacktrace:
  [1] error(s::String)
    @ Base ./error.jl:44
  [2] top-level scope
    @ ~/.julia/packages/CPLEX/5jmjD/src/CPLEX.jl:12
  [3] include(mod::Module, _path::String)
    @ Base ./Base.jl:306
  [4] include_package_for_output(pkg::Base.PkgId, input::String, depot_path::Vector{String}, dl_load_path::Vector{String}, 

# Miexed integer programming

# Young couple

Formulate a binary mixed integer model that minimizes the total number of required
hours for household jobs and ensures that all jobs are done by someone

In [11]:
using JuMP, HiGHS

processes = 1:4
people = 1:2

hours_per_task = [
    4.5 7.8 3.6 2.9;
    4.9 7.2 4.3 3.1
]

model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

# Each process is a binary variable for each persons
@variable(model, x[people, processes], Bin)

# Each process is assigned to exactly one person
@constraint(model, [j in processes], sum(x[i, j] for i in people) == 1)


# Minimize total hours
@objective(model, Min, sum(hours_per_task[i, j] * x[i, j] for i in people, j in processes))

optimize!(model)

println("Optimal solution:")
println("z = ", (objective_value(model)))

println("Assignments: ", (value.(x)))

Optimal solution:
z = 18.2
Assignments: 2-dimensional DenseAxisArray{Float64,2,...} with index sets:
    Dimension 1, 1:2
    Dimension 2, 1:4
And data, a 2×4 Matrix{Float64}:
 1.0  0.0  1.0  1.0
 0.0  1.0  0.0  0.0


But Eve does 3 task and Steven does 1, we should balance this so they make the same no. of task

In [13]:
using JuMP, HiGHS

processes = 1:4
people = 1:2

hours_per_task = [
    4.5 7.8 3.6 2.9;
    4.9 7.2 4.3 3.1
]

model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

# Each process is a binary variable for each persons
@variable(model, x[people, processes], Bin)

# Each process is assigned to exactly one person
@constraint(model, [j in processes], sum(x[i, j] for i in people) == 1)
@constraint(model, [i in people], sum(x[i, j] for j in processes) == length(processes) / length(people))

# Minimize total hours
@objective(model, Min, sum(hours_per_task[i, j] * x[i, j] for i in people, j in processes))

optimize!(model)

println("Optimal solution:")
println("z = ", (objective_value(model)))

println("Assignments: ", (value.(x)))

Optimal solution:
z = 18.4
Assignments: 2-dimensional DenseAxisArray{Float64,2,...} with index sets:
    Dimension 1, 1:2
    Dimension 2, 1:4
And data, a 2×4 Matrix{Float64}:
  1.0  0.0   1.0  0.0
 -0.0  1.0  -0.0  1.0


# Startup fund

In [15]:
profit = [29 35 24 52 53 41 43 68 28]
capital = [17 25 19 25 28 23 29 31 18]

model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

@variable(model, x[1:9], Bin)


@objective(model, Max, sum(profit[i] * x[i] for i in 1:9))

@constraint(model, sum(capital[i] * x[i] for i in 1:9) <= 100)

optimize!(model)

println("Optimal solution:")
println("z = ", (objective_value(model)))
println("Investments: ", (value.(x)))


Optimal solution:
z = 191.0
Investments: [1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0]


In [23]:
profit = [29 35 24 52 53 41 43 68 28]
capital = [17 25 19 25 28 23 29 31 18]

model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

@variable(model, x[1:9], Bin)


@objective(model, Max, sum(profit[i] * x[i] for i in 1:9))

@constraint(model, sum(capital[i] * x[i] for i in 1:9) <= 100)

@constraint(model, x[1] + x[5] <= 1)

# 6 and 9 require (2 OR 3)
@constraint(model, x[6] <= x[2] + x[3])
@constraint(model, x[9] <= x[2] + x[3])


optimize!(model)

println("Optimal solution:")
println("z = ", (objective_value(model)))
println("Investments: ", (value.(x)))


Optimal solution:
z = 185.0
Investments: [-0.0, -0.0, 1.0, 1.0, 0.0, 1.0, -0.0, 1.0, 0.0]


# Logic

# Stamps

In [ ]:
using JuMP, HiGHS
include("Data/stamp_bid_data.jl")

println("Sizes of data structures:")
println(size(BidPrice))  
println(size(BidSets))  

noStamps = size(BidSets, 2) # Columns
noBids   = size(BidSets, 1) # Rows


model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

@variable(model, x[1:noBids], Bin)

@objective(model, Max, sum(BidPrice[b] * x[b] for b in 1:noBids))

# We loop through all stamps
for s in 1:noStamps
    # For each stamp, the sum of all bids containing that stamp
    # that are accepted must be at most 1
    @constraint(model, sum(BidSets[b, s] * x[b] for b in 1:noBids) <= 1)
end

optimize!(model)

println("\nOptimal solution:")
println("z = ", (objective_value(model)))
println("No. of bids accepted: ", (sum(value.(x))))

Sizes of data structures:
(1, 213)
(213, 100)

Optimal solution:
z = 11084.0
No. of bids accepted: 2.0


In [66]:
using JuMP, HiGHS
include("Data/stamp_bid_data.jl")

println("Sizes of data structures:")
println(size(BidPrice))  
println(size(BidSets))  

noStamps = size(BidSets, 2) # Columns
noBids   = size(BidSets, 1) # Rows


model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

@variable(model, x[1:noBids], Bin)

@objective(model, Max, sum(BidPrice[b] * x[b] for b in 1:noBids))

# Make sure that a stamp is sold at most once
for s in 1:noStamps
    @constraint(model, sum(BidSets[b, s] * x[b] for b in 1:noBids) == 1)
end

optimize!(model)

println("\nOptimal solution:")
println("z = ", (objective_value(model)))
println("No. of bids accepted: ", (sum(value.(x))))

Sizes of data structures:
(1, 213)
(213, 100)

Optimal solution:


MathOptInterface.ResultIndexBoundsError{MathOptInterface.ObjectiveValue}: Result index of attribute MathOptInterface.ObjectiveValue(1) out of bounds. There are currently 0 solution(s) in the model.